# Training Deteksi Kendaraan Indonesia (YOLO11)

Dataset: `puteri-marlisajasmine/deteksi-kendaraan-indonesia-3-ddnbr` versi 3 (Roboflow).

**PENTING sebelum mulai:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Tanpa GPU, training akan sangat lambat.

Jalankan sel berurutan dari atas. Total ± 30-60 menit untuk 100 epoch di T4.

## 1. Install dependensi

**Jalankan sel ini paling pertama**, sebelum sel apa pun yang meng-`import torch`.


In [ ]:
%pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 2. Cek GPU

Dijalankan **setelah** install, karena `pip install` mengganti versi torch di disk.
Kalau `import torch` dijalankan sebelum install, torch jadi setengah ter-load dan muncul
`AttributeError: partially initialized module 'torch'` -> obatnya Runtime > Mulai ulang sesi.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU tidak aktif! Runtime -> Change runtime type -> T4 GPU'
print('GPU siap:', torch.cuda.get_device_name(0))

## 3. Unduh dataset dari Roboflow

API key diminta lewat prompt (tidak tersimpan di notebook, jadi aman kalau notebook dibagikan).
Ambil di Roboflow → Settings → API Keys → **Private API Key**.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass('Roboflow Private API Key: ')

rf = Roboflow(api_key=API_KEY)
version = rf.workspace('puteri-marlisajasmine').project('deteksi-kendaraan-indonesia-3-ddnbr').version(3)
try:
    ds = version.download('yolov11', location='/content/dataset', overwrite=True)
except Exception as e:
    print('format yolov11 gagal:', e, '-> fallback yolov8')
    ds = version.download('yolov8', location='/content/dataset', overwrite=True)

DATA_YAML = f'{ds.location}/data.yaml'
print('\ndata.yaml:', DATA_YAML)
print(open(DATA_YAML).read())

In [ ]:
# Cek jumlah gambar tiap split
import glob, yaml
cfg = yaml.safe_load(open(DATA_YAML))
print('kelas:', cfg['names'])
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{ds.location}/{split}/images/*'))
    print(f'{split:6s}: {n} gambar')

## 4. Training

`yolo11s` = titik seimbang akurasi/kecepatan. Ganti ke `yolo11m.pt` kalau mau lebih akurat
(di T4 masih muat, batch turunkan ke 8), atau `yolo11n.pt` kalau target deploy di perangkat lemah.

In [ ]:
from ultralytics import YOLO

MODEL   = 'yolo11s.pt'
EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16      # T4 16GB: 16 aman untuk yolo11s @640

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,          # Colab cuma 2 vCPU
    amp=True,           # aman di CUDA, ~2x lebih cepat
    patience=30,        # early stopping
    project='/content/runs',
    name='kendaraan',
    exist_ok=True,
    seed=0,
    plots=True,
    # augmentasi khas kamera jalan raya
    degrees=0.0,        # kamera CCTV jarang miring
    fliplr=0.5,
    scale=0.5,          # kendaraan dekat vs jauh
    mosaic=1.0,
    close_mosaic=10,
)

## 5. Evaluasi di test set

In [ ]:
best = YOLO('/content/runs/kendaraan/weights/best.pt')
m = best.val(data=DATA_YAML, split='test', device=0)

print(f'\nmAP50    : {m.box.map50:.4f}')
print(f'mAP50-95 : {m.box.map:.4f}\n')
print(f"{'kelas':<16}{'P':>8}{'R':>8}{'mAP50':>9}")
for i, c in enumerate(m.box.ap_class_index):
    print(f'{best.names[c]:<16}{m.box.p[i]:>8.3f}{m.box.r[i]:>8.3f}{m.box.ap50[i]:>9.3f}')

In [ ]:
# Kurva training + confusion matrix
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    p = f'/content/runs/kendaraan/{f}'
    import os
    if os.path.exists(p):
        print(f)
        display(Image(filename=p, width=900))

## 6. Ambil bobot hasil training

Jalankan **salah satu** sel di bawah. Setelah `best.pt` ada di Mac, taruh di
`~/vehicle-counter/runs/kendaraan/weights/best.pt` lalu jalankan `src/count.py` seperti biasa.

In [ ]:
# Opsi A: download langsung ke Mac (browser akan menyimpan ke folder Downloads)
from google.colab import files
files.download('/content/runs/kendaraan/weights/best.pt')

In [ ]:
# Opsi B: simpan ke Google Drive (lebih aman kalau runtime keburu putus)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/vehicle-counter'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/runs/kendaraan/weights/best.pt', f'{dst}/best.pt')
shutil.make_archive(f'{dst}/runs_kendaraan', 'zip', '/content/runs/kendaraan')
print('tersimpan di Drive:', dst)

## 7. (Opsional) Uji cepat di video

Upload satu video jalan raya lewat panel Files di kiri, lalu jalankan sel ini untuk
memastikan deteksi + ByteTrack sudah masuk akal sebelum dipakai menghitung di Mac.

In [ ]:
VIDEO = '/content/jalan.mp4'   # sesuaikan nama file yang kamu upload

import collections
ids_per_kelas = collections.defaultdict(set)

for r in best.track(source=VIDEO, tracker='bytetrack.yaml', persist=True,
                    conf=0.3, imgsz=960, device=0, stream=True,
                    save=True, project='/content/runs', name='predict', exist_ok=True):
    b = r.boxes
    if b is not None and b.id is not None:
        for tid, c in zip(b.id.int().tolist(), b.cls.int().tolist()):
            ids_per_kelas[best.names[c]].add(tid)

print('Perkiraan kendaraan unik (jumlah track ID):')
for k, v in sorted(ids_per_kelas.items()):
    print(f'  {k:<14}{len(v)}')
print('\nVideo hasil: /content/runs/predict/')
print('Catatan: ini hitungan kasar per track ID. Penghitungan resmi berbasis')
print('garis lintasan ada di src/count.py yang jalan di Mac.')